In [3]:
import ase
import numpy as np
import pyscf
import time
import os

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
from pyscf import gto, dft, df, lib
from pyscf.scf import hf
import scipy
hf.MUTE_CHKFILE = True

#mol = gto.M(atom='O  0  0  0.1184; H  0,  0.7532, -0.4735; H 0,  -0.7532, -0.4735 ', basis='def2svp')
#mf = dft.RKS(mol)
#mf.chkfile=False
#mf.xc = 'pbe'
#mf.kernel()
#g = mf.nuc_grad_method()
#g.kernel()
basis = 'augccpvdz'
auxbasis = 'augccpvdzjkfit'
atoms = ['H', 'O', 'C', 'N', 'S']
atom_nums = [1, 8, 6, 7, 16]
save_path_calc = '_augccpvdz_calc_df_augccpvdzjkfit.npy'
save_path = '_augccpvdz.npy'
for i, atom_type in enumerate(atoms):
    print('calc', atom_type)
    start = time.time()
    if atom_nums[i]%2 == 1:
        spin = 1
    else:
        spin = 0
    mol = gto.M(atom=[(atom_type, [0, 0, 0])], basis=basis, spin=spin)
    #print(mol.pack())
    mf = dft.RKS(mol)
    mf.chkfile=False
    mf.xc = 'pbe'
    mf.kernel()
    g = mf.nuc_grad_method()
    forces = g.grad()
    print('elapsed', time.time() - start)
    #print(mfs[i].mo_coeff)
    res = []
    res.append(mol.pack())
    calc_dict = {}
    print('mo occ', mf.mo_occ)
    calc_dict['mo_coeff'] = mf.mo_coeff
    calc_dict['mo_occ'] = mf.mo_occ
    calc_dict['energy'] = mf.e_tot
    calc_dict['forces'] = forces
                      
    dm1 = mf.make_rdm1(mf.mo_coeff, mf.mo_occ)

    # Define the auxiliary fitting basis for 3-center integrals. Use the function
    # make_auxmol to construct the auxiliary Mole object (auxmol) which will be
    # used to generate integrals.
    auxmol = df.addons.make_auxmol(mol, auxbasis)

    # ints_3c is the 3-center integral tensor (ij|P), where i and j are the
    # indices of AO basis and P is the auxiliary basis
    ints_3c2e = df.incore.aux_e2(mol, auxmol, intor='int3c2e')
    ints_2c2e = auxmol.intor('int2c2e')
    print('ints3c2e shape', ints_3c2e.shape)
    print('ints2c2e shape', ints_2c2e.shape)

    nao = mol.nao
    naux = auxmol.nao

    # Compute the DF coefficients (df_coef) and the DF 2-electron (df_eri)
    df_coef = scipy.linalg.solve(ints_2c2e, ints_3c2e.reshape(nao*nao, naux).T)
    df_coef = df_coef.reshape(naux, nao, nao)
    print('df coef shape', df_coef.shape)
    print('dm1 shape', dm1.shape)
    #print('dm1', dm1)
    if dm1.ndim > 2:
        df_basis = []
        for j in range(dm1.shape[0]):
            df_basis.append(lib.einsum('Pij,ij->P', df_coef, dm1[j]))
        df_basis = np.stack(df_basis, axis=0)
        print(df_basis.shape)
            
    else:
        df_basis = lib.einsum('Pij,ij->P', df_coef, dm1)

    mol.basis = auxbasis 
    mol.build()
    calc_dict['df_coeff'] = df_basis
    calc_dict['auxbasis'] = auxbasis
    res.append(calc_dict)
    
    atoms = {}
    atoms['positions'] = np.array([[[0, 0, 0]]])
    atoms['atom_numbers'] = np.array([atom_nums[i]])
    atoms['atom_types'] = np.array([atom_type])
    np.save('datasets/' + atom_type + save_path_calc, [res], allow_pickle=True)
    np.save('datasets/' + atom_type + save_path, atoms, allow_pickle=True)

calc H
converged SCF energy = -0.499224422431329  <S^2> = 0.75  2S+1 = 2
--------------- UKS gradients ---------------
         x                y                z
0 H     0.0000000000     0.0000000000     0.0000000000
----------------------------------------------
elapsed 0.09980344772338867
mo occ [[1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0.]]
ints3c2e shape (9, 9, 32)
ints2c2e shape (32, 32)
df coef shape (32, 9, 9)
dm1 shape (2, 9, 9)
(2, 32)
calc O
SCF not converged.
SCF energy = -74.8813367024845
--------------- RKS gradients ---------------
         x                y                z
0 O    -0.0000000000     0.0000000000     0.0000000000
----------------------------------------------
elapsed 0.4766416549682617
mo occ [2. 2. 2. 2. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
ints3c2e shape (23, 23, 86)
ints2c2e shape (86, 86)
df coef shape (86, 23, 23)
dm1 shape (23, 23)
calc C
SCF not converged.
SCF energy = -37.7190475845007
--------------- RKS gradi

In [5]:
import torch
from equiv_dens.training.parse_command_line_arguments import parse_command_line_arguments
from equiv_dens.training.errors import ErrorDict
from equiv_dens.data.density_dataset import AtomsDensityData
from equiv_dens.data.hamiltonian_dataset import seeded_random_split
from equiv_dens.utils.grids import cubical_grid, cubical_sampling,\
    dftpy_grid, CubicalGrid, spherical_grid, spherical_radial_sampling
from equiv_dens.training.model_loader import load_model
import equiv_dens.utils.base as utils
from equiv_dens.utils import orbitals
from functools import partial

basis = 'augccpvdz'
auxbasis = 'augccpvqzjkfit'
atoms = ['H', 'O', 'C', 'N', 'S']
atom_nums = [1, 8, 6, 7, 16]
save_path_calc = '_augccpvdz_calc_df_augccpvqzjkfit.npy'
save_path = '_augccpvdz.npy'
for i, atom_type in enumerate(atoms):
    np_path = 'datasets/' + atom_type + save_path
    dens_path = 'datasets/' + atom_type + save_path_calc
    grid_fn = partial(spherical_grid, level=2)
    sampling_fn = partial(spherical_radial_sampling, rotate=False)
    grid_origin = 0
    grid_extent = None
    
    dataset = AtomsDensityData(np_path=np_path, density_path=dens_path,
                               orbitals_path='datasets/augccpvqzjkfit_orbital_basis_df.npy',
                               density_n_samp=10000000000,
                               required_properties=['density'],
                               center_positions=False,
                               radial_coeffs_file='datasets/augccpvqzjkfit_radial_coeffs_df.npy',
                               dtype=torch.float32,
                               grid_fn=grid_fn,
                               sampling_fn=sampling_fn,
                               grid_extent=grid_extent,
                               grid_origin=grid_origin,
                               verbose=0,
                               radii_adjust=True)
    
    dataset_df = AtomsDensityData(np_path=np_path, density_path=dens_path,
                               orbitals_path='datasets/augccpvqzjkfit_orbital_basis_df.npy',
                               density_n_samp=10000000000,
                               required_properties=['density'],
                               center_positions=False,
                               radial_coeffs_file='datasets/augccpvqzjkfit_radial_coeffs_df.npy',
                               dtype=torch.float32,
                               grid_fn=grid_fn,
                               sampling_fn=sampling_fn,
                               grid_extent=grid_extent,
                               grid_origin=grid_origin,
                               verbose=0,
                               radii_adjust=True,
                               projected_density=True)
    
    samp = dataset.get_properties(0)
    samp_df = dataset_df.get_properties(0)
    print('density loss', torch.sum(torch.abs(samp['density'] - samp_df['density']) * samp['coord_weights'])/samp['atom_numbers'])
    print('density integral', torch.sum(samp['density'] * samp['coord_weights']))
    print('df integral', torch.sum(samp_df['density'] * samp['coord_weights']))
    print(dataset_df.density_fitting)
    # print('grid_spec', dataset.grid_spec)

Use "numpy" for Fourier Transform
Starting atomsdata density init
Some variables
atoms keys dict_keys(['positions', 'atom_numbers', 'atom_types'])
grid fn functools.partial(<function spherical_grid at 0x7f62341c87b8>, level=2)
len atom types 1
atom numbers 1
level 2
finished init
Starting atomsdata density init
Some variables
atoms keys dict_keys(['positions', 'atom_numbers', 'atom_types'])
grid fn functools.partial(<function spherical_grid at 0x7f62341c87b8>, level=2)
len atom types 1
atom numbers 1
level 2
finished init
density loss tensor([[0.0125]])
density integral tensor(1.0000)
df integral tensor(0.9999)
[{'df_coeff': array([[ 2.83307149e-02,  8.03729363e-02,  8.38720361e-02,
         2.94724356e-02,  2.05145448e-03,  4.58785862e-20,
         4.58236698e-20, -7.17370218e-20,  3.92874629e-19,
         4.77235947e-19,  2.39810554e-19,  1.74734621e-19,
         1.43612445e-19, -5.84465178e-19,  3.48913680e-20,
         1.43892162e-20, -2.60560903e-19,  5.07889120e-38,
         2.00

In [6]:
import torch
from equiv_dens.training.parse_command_line_arguments import parse_command_line_arguments
from equiv_dens.training.errors import ErrorDict
from equiv_dens.data.density_dataset import AtomsDensityData
from equiv_dens.data.hamiltonian_dataset import seeded_random_split
from equiv_dens.utils.grids import cubical_grid, cubical_sampling,\
    dftpy_grid, CubicalGrid, spherical_grid, spherical_radial_sampling
from equiv_dens.training.model_loader import load_model
import equiv_dens.utils.base as utils
from equiv_dens.utils import orbitals
from functools import partial

basis = 'augccpvdz'
auxbasis = 'augccpvqzjkfit'
atoms = ['H', 'O', 'C', 'N', 'S']
atom_nums = [1, 8, 6, 7, 16]
save_path_calc = '_augccpvdz_calc_df_augccpvdzjkfit.npy'
save_path = '_augccpvdz.npy'
for i, atom_type in enumerate(atoms):
    np_path = 'datasets/' + atom_type + save_path
    dens_path = 'datasets/' + atom_type + save_path_calc
    grid_fn = partial(spherical_grid, level=2)
    sampling_fn = partial(spherical_radial_sampling, rotate=False)
    grid_origin = 0
    grid_extent = None
    
    dataset = AtomsDensityData(np_path=np_path, density_path=dens_path,
                               orbitals_path='datasets/augccpvqzjkfit_orbital_basis_df.npy',
                               density_n_samp=10000000000,
                               required_properties=['density'],
                               center_positions=False,
                               radial_coeffs_file='datasets/augccpvqzjkfit_radial_coeffs_df.npy',
                               dtype=torch.float32,
                               grid_fn=grid_fn,
                               sampling_fn=sampling_fn,
                               grid_extent=grid_extent,
                               grid_origin=grid_origin,
                               verbose=0,
                               radii_adjust=True)
    
    dataset_df = AtomsDensityData(np_path=np_path, density_path=dens_path,
                               orbitals_path='datasets/augccpvqzjkfit_orbital_basis_df.npy',
                               density_n_samp=10000000000,
                               required_properties=['density'],
                               center_positions=False,
                               radial_coeffs_file='datasets/augccpvqzjkfit_radial_coeffs_df.npy',
                               dtype=torch.float32,
                               grid_fn=grid_fn,
                               sampling_fn=sampling_fn,
                               grid_extent=grid_extent,
                               grid_origin=grid_origin,
                               verbose=0,
                               radii_adjust=True,
                               projected_density=True)
    
    samp = dataset.get_properties(0)
    samp_df = dataset_df.get_properties(0)
    print('density loss', torch.sum(torch.abs(samp['density'] - samp_df['density']) * samp['coord_weights'])/samp['atom_numbers'])
    print('density integral', torch.sum(samp['density'] * samp['coord_weights']))
    print('df integral', torch.sum(samp_df['density'] * samp['coord_weights']))
    print(dataset_df.density_fitting)
    # print('grid_spec', dataset.grid_spec)

Starting atomsdata density init
Some variables
atoms keys dict_keys(['positions', 'atom_numbers', 'atom_types'])
grid fn functools.partial(<function spherical_grid at 0x7f62341c87b8>, level=2)
len atom types 1
atom numbers 1
level 2
finished init
Starting atomsdata density init
Some variables
atoms keys dict_keys(['positions', 'atom_numbers', 'atom_types'])
grid fn functools.partial(<function spherical_grid at 0x7f62341c87b8>, level=2)
len atom types 1
atom numbers 1
level 2
finished init
density loss tensor([[0.0125]])
density integral tensor(1.0000)
df integral tensor(0.9999)
[{'df_coeff': array([[ 2.83307149e-02,  8.03729363e-02,  8.38720361e-02,
         2.94724356e-02,  2.05145448e-03, -3.23348631e-20,
         1.04242032e-20, -1.54777016e-20, -5.14277988e-19,
         7.21161549e-19,  7.59679042e-19, -3.66480919e-20,
        -1.90563638e-19, -3.84077963e-19,  3.26628631e-20,
        -1.44478018e-19, -2.26964090e-19, -7.62372449e-37,
         5.31204801e-37, -2.71472642e-37, -6.96

In [7]:
args, hyperparam_args = parse_command_line_arguments(arg_file='C_dens.txt')

print('type dtype', type(args.dtype))
args.fix_arguments = True
print('args np dir', args.np_dataset)
# no restart directory specified
directory = args.restart  # load directory name
# load latest checkpoint
checkpoint_path = os.path.join(directory, 'checkpoints')  # checkpoint directory
checkpoint = torch.load(os.path.join(
    checkpoint_path, 'latest_checkpoint.pth'), map_location='cpu')
latest_checkpoint = checkpoint['step']
model_code = checkpoint['ID']  # load ID
step = checkpoint['step']
for arg in vars(checkpoint['args']):
    if args.fix_arguments:
        if arg in hyperparam_args:
            print('loading hyperparam arg', arg)
            setattr(args, arg, getattr(checkpoint['args'], arg))
    else:
        print('loading all arg', arg)
        setattr(args, arg, getattr(checkpoint['args'], arg))
restore = True

args.best_model_path = 'best_' + model_code + '.pth'
print('best_model_path', args.best_model_path)

print('model code:', model_code)
# determine whether GPU is used for training
print('args use gpu', args.use_gpu)
args.use_gpu = False
# load dataset(s)
print("loading density from" + str(args.dens_dataset) + "...")
print("loading atoms from" + args.np_dataset + "...")

args.verbose = 0
args.use_gpu = False
args.radii_adjust = True 
grid_fn = partial(spherical_grid, level=2)
sampling_fn = partial(spherical_radial_sampling, rotate=False)
grid_origin = 0
grid_extent = None
    
dataset = AtomsDensityData(np_path=args.np_dataset, density_path=args.dens_dataset,
                           orbitals_path=args.orbitals_file,
                           density_n_samp=10000000000,
                           required_properties=['density'],
                           center_positions=False,
                           radial_coeffs_file=args.radial_coeffs_file,
                           dtype=torch.float32,
                           grid_fn=grid_fn,
                           sampling_fn=sampling_fn,
                           grid_extent=grid_extent,
                           grid_origin=grid_origin,
                           verbose=0,
                           radii_adjust=args.radii_adjust)

dataset_df = AtomsDensityData(np_path=args.np_dataset, density_path=args.dens_dataset,
                           orbitals_path=args.orbitals_file,
                           density_n_samp=10000000000,
                           required_properties=['density'],
                           center_positions=False,
                           radial_coeffs_file=args.radial_coeffs_file,
                           dtype=torch.float32,
                           grid_fn=grid_fn,
                           sampling_fn=sampling_fn,
                           grid_extent=grid_extent,
                           grid_origin=grid_origin,
                           verbose=0,
                           radii_adjust=args.radii_adjust,
                           projected_density=True)
    
samp = dataset.get_properties(0)
samp_df = dataset_df.get_properties(0)
print('density loss', torch.sum(torch.abs(samp['density'] - samp_df['density']) * samp['coord_weights'])/samp['atom_numbers'])
print('density integral', torch.sum(samp['density'] * samp['coord_weights']))
print('df integral', torch.sum(samp_df['density'] * samp['coord_weights']))
print(dataset_df.density_fitting)
print(dataset.atoms)
model = load_model(args, dataset)

usage: ipykernel_launcher.py [-h] [--restart FOLDER] [--load_from STR]
                             [--fix_arguments True|False] [--activation STR]
                             [--order INT [INT ...]]
                             [--mixing_order INT [INT ...]]
                             [--order_en INT [INT ...]]
                             [--mixing_order_en INT [INT ...]]
                             [--num_features INT] [--num_basis_functions INT]
                             [--num_radial_components INT]
                             [--num_energy_features INT] [--num_modules INT]
                             [--num_en_modules INT] [--num_residual_pre_x INT]
                             [--num_residual_post_x INT]
                             [--num_residual_pre_vi INT]
                             [--num_residual_pre_vj INT]
                             [--num_residual_post_v INT]
                             [--num_residual_output INT]
                             [--num_energy

SystemExit: 2

/home/mihail/anaconda3/envs/equiv_dens/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3425: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
